# Mitsui Commodity Prediction Challenge

This notebook contains the complete pipeline for training and evaluating models for the Mitsui Commodity Prediction Challenge.

Key features:
- Time-series aware training with LightGBM
- Comprehensive timing analysis system
- Public leaderboard simulation (last 90 days holdout)
- Cross-validation with time-series splits
- Model packaging and artifact management
- Kaggle inference server integration

In [ ]:
# Core Imports
import os
import numpy as np
import pandas as pd
import polars as pl
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
import time
from collections import defaultdict
from pathlib import Path
import zipfile

In [ ]:
# Configuration Settings
TRAIN = True      # True: train + (optional) CV + save; False: load + predict
DO_CV = True      # CV is only used when TRAIN=True
CV_SPLITS = 3     # Number of cross-validation splits
DO_PUBLIC90 = True # Execute end-90-day holdout validation

# Environment detection and path configuration
IS_KAGGLE = os.getenv('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_COMP_RUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None

if IS_COMP_RUN:
    TRAIN = False
    DO_CV = False
    DO_PUBLIC90 = False

if IS_KAGGLE:
    DATA_PATH = '/kaggle/input/mitsui-commodity-prediction-challenge/'
    MODEL_INPUT_DIR = '/kaggle/input/Input_dir/'
    MODEL_OUTPUT_DIR = '/kaggle/working/model'
else:
    DATA_PATH = './'
    MODEL_INPUT_DIR = './model/'
    MODEL_OUTPUT_DIR = './model'

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="darkgrid")

In [ ]:
# Enhanced Time Analysis System
timing_data = defaultdict(list)  # Store all timing measurements
step_times = {}  # Store step start times
pipeline_start = time.time()

def start_timer(process_name):
    """Start timing a process"""
    step_times[process_name] = time.time()

def end_timer(process_name):
    """End timing a process and store the duration"""
    if process_name in step_times:
        duration = time.time() - step_times[process_name]
        timing_data[process_name].append(duration)
        del step_times[process_name]
        return duration
    return 0

def plot_timing_histogram(process_name, title=None, bins=20):
    """Plot histogram for a specific process timing"""
    if process_name not in timing_data or len(timing_data[process_name]) == 0:
        print(f"No timing data available for {process_name}")
        return
    
    times = timing_data[process_name]
    plt.figure(figsize=(10, 6))
    plt.hist(times, bins=min(bins, len(times)), alpha=0.7, edgecolor='black')
    plt.title(title or f'Time Distribution: {process_name}')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    
    # Add statistics
    mean_time = np.mean(times)
    std_time = np.std(times)
    plt.axvline(mean_time, color='red', linestyle='--', label=f'Mean: {mean_time:.3f}s')
    plt.axvline(mean_time + std_time, color='orange', linestyle='--', alpha=0.7, label=f'Mean+Std: {mean_time+std_time:.3f}s')
    plt.axvline(mean_time - std_time, color='orange', linestyle='--', alpha=0.7, label=f'Mean-Std: {mean_time-std_time:.3f}s')
    plt.legend()
    
    print(f"\n{process_name} Statistics:")
    print(f"  Count: {len(times)}")
    print(f"  Mean: {mean_time:.3f}s")
    print(f"  Std: {std_time:.3f}s")
    print(f"  Min: {np.min(times):.3f}s")
    print(f"  Max: {np.max(times):.3f}s")
    print(f"  Total: {np.sum(times):.3f}s")
    
    plt.tight_layout()
    plt.show()

start_timer("setup")

In [ ]:
# Model Performance Tracking System
class ModelPerformanceTracker:
    def __init__(self):
        self.training_scores = {}  # target -> training R2/MSE
        self.validation_scores = {}  # target -> validation scores
        self.cv_scores_per_target = defaultdict(list)  # target -> list of CV scores
        self.prediction_times = defaultdict(list)
        self.model_sizes = {}  # target -> model size in MB
        
    def record_training_score(self, target, score, metric='r2'):
        if target not in self.training_scores:
            self.training_scores[target] = {}
        self.training_scores[target][metric] = score
    
    def record_validation_score(self, target, score, metric='sharpe'):
        if target not in self.validation_scores:
            self.validation_scores[target] = {}
        self.validation_scores[target][metric] = score
    
    def get_model_size(self, model):
        import pickle
        return len(pickle.dumps(model)) / (1024 * 1024)  # Size in MB
    
    def plot_performance_comparison(self):
        # Implementation below
        pass

perf_tracker = ModelPerformanceTracker()

In [ ]:
# Data Loading and Initial Processing
start_timer("data_loading")
print("Loading data...")
train_df = pl.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test_df = pl.read_csv(os.path.join(DATA_PATH, 'test.csv'))
train_labels = pl.read_csv(os.path.join(DATA_PATH, 'train_labels.csv'))
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}, Labels shape: {train_labels.shape}")
end_timer("data_loading")

start_timer("data_sorting")
# Temporal leak prevention: ensure data is properly sorted by date
if 'date_id' in train_df.columns:
    train_df = train_df.sort('date_id')
if 'date_id' in train_labels.columns:
    train_labels = train_labels.sort('date_id')
end_timer("data_sorting")

In [ ]:
# Feature and Target Column Definitions
start_timer("feature_definition")
EXCLUDE_COLS = {'date_id', 'row_id', 'is_scored'}
COMMON_COLUMNS = [c for c in train_df.columns if c in set(test_df.columns)]
FEATURE_BASE = [c for c in COMMON_COLUMNS if c not in EXCLUDE_COLS]
TARGETS = [f"target_{i}" for i in range(424)]
end_timer("feature_definition")
end_timer("setup")

In [ ]:
# Data Preprocessing Functions
def preprocess_for_lgbm(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare DataFrame for LightGBM by handling data types and missing values.
    Converts object columns to numeric and categorical codes to integers.
    """
    start_timer("preprocess_for_lgbm_single")
    
    df = df.copy()
    # Handle specific columns that may have object dtype but should be numeric
    for col in [
        'US_Stock_GOLD_adj_open','US_Stock_GOLD_adj_high',
        'US_Stock_GOLD_adj_low','US_Stock_GOLD_adj_close',
        'US_Stock_GOLD_adj_volume'
    ]:
        if col in df and df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert categorical columns to integer codes
    for cat in df.select_dtypes(['category']).columns:
        df[cat] = df[cat].cat.codes
    
    # Replace infinite values with NaN for better model handling
    df = df.replace([np.inf, -np.inf], np.nan)
    
    end_timer("preprocess_for_lgbm_single")
    return df

def create_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Feature engineering pipeline - currently a placeholder that returns input unchanged.
    This function can be extended to add technical indicators, rolling statistics, etc.
    """
    start_timer("create_features_single")
    result = df.clone()
    end_timer("create_features_single")
    return result

In [ ]:
# Evaluation Metrics
SOLUTION_NULL_FILLER = -999999

def rankcorr_sharpe(preds_df: pd.DataFrame, truths_df: pd.DataFrame, filler: float = SOLUTION_NULL_FILLER) -> float:
    """
    Calculate competition metric: Sharpe ratio of daily Spearman rank correlations.
    For each date, compute rank correlation across all targets between predictions and labels.
    Return the Sharpe ratio (mean/std) of these daily correlations.
    """
    start_timer("rankcorr_sharpe_single")
    
    P = preds_df.copy().fillna(0.0)
    T = truths_df.copy().fillna(filler)
    daily_scores = []
    
    for p_row, t_row in zip(P.values, T.values):
        # Only consider non-filler values with finite predictions
        mask = (t_row != filler) & np.isfinite(p_row)
        if mask.sum() < 2:
            daily_scores.append(0.0)
            continue
        
        # Calculate Spearman rank correlation
        p_rank = pd.Series(p_row[mask]).rank(method='average').to_numpy()
        t_rank = pd.Series(t_row[mask]).rank(method='average').to_numpy()
        
        if np.std(p_rank) == 0 or np.std(t_rank) == 0:
            daily_scores.append(0.0)
        else:
            daily_scores.append(np.corrcoef(p_rank, t_rank)[0, 1])
    
    arr = np.asarray(daily_scores, dtype=float)
    std = arr.std(ddof=0)
    result = float(arr.mean() / std) if std > 0 else 0.0
    
    end_timer("rankcorr_sharpe_single")
    return result

def validate_public_90_style(train_pl: pl.DataFrame, labels_pl: pl.DataFrame) -> float:
    """
    Simulate Kaggle public leaderboard evaluation by holding out the last 90 days.
    This provides a more realistic estimate of model performance.
    """
    start_timer("validate_public_90_style")
    
    X_all_df = create_features(train_pl).to_pandas()
    X_all = X_all_df[FEATURE_BASE].copy()
    y_all = labels_pl.to_pandas()[TARGETS].copy()
    N = len(X_all)
    H = 90
    
    if N <= H:
        print("[Public-90] Not enough length; skipped.")
        end_timer("validate_public_90_style")
        return 0.0
    
    # Split indices: train on first N-90 days, validate on last 90
    tr_idx = np.arange(0, N - H)
    va_idx = np.arange(N - H, N)
    preds_va = pd.DataFrame(index=range(H), columns=TARGETS, dtype=float)
    sol_va = y_all.iloc[va_idx].reset_index(drop=True)
    
    # Train separate model for each target
    for idx, tgt in enumerate(TARGETS):
        start_timer("public90_single_target_training")
        
        y = y_all[tgt]
        mask_tr = (~y.isna()).values & (np.arange(N) < (N - H))
        
        if mask_tr.sum() == 0:
            preds_va[tgt] = 0.0
            end_timer("public90_single_target_training")
            continue
        
        # Prepare training and validation sets with target encoding
        Xtr = X_all.loc[mask_tr].copy()
        ytr = y.loc[mask_tr]
        Xva = X_all.iloc[va_idx].copy()
        Xtr['target_name_encoded'] = idx
        Xva['target_name_encoded'] = idx
        
        Xtr = preprocess_for_lgbm(Xtr).reindex(columns=(FEATURE_BASE + ['target_name_encoded']), fill_value=0.0)
        Xva = preprocess_for_lgbm(Xva).reindex(columns=(FEATURE_BASE + ['target_name_encoded']), fill_value=0.0)
        
        # Train LightGBM model
        m = LGBMRegressor(
            n_estimators=100, learning_rate=0.05,
            max_depth=-1, num_leaves=64,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbose=-1
        )
        m.fit(Xtr, ytr)
        preds_va[tgt] = m.predict(Xva)
        
        end_timer("public90_single_target_training")
    
    score = rankcorr_sharpe(preds_va, sol_va, filler=SOLUTION_NULL_FILLER)
    end_timer("validate_public_90_style")
    return float(score)

In [ ]:
# Model Storage and Metadata Management
models = {}  # target -> LightGBM model
MODEL_FEATURES = {}  # target -> list of feature names used at train time (order matters)
MODELS_LOADED = False

def _meta_path_for(tgt: str) -> str:
    """Generate file path for storing feature metadata for a target."""
    return os.path.join(MODEL_OUTPUT_DIR, f"{tgt}_feat.pkl")

def _load_model_and_meta(tgt: str, input_dir: str):
    """Load both model and feature metadata for a specific target."""
    start_timer("model_loading_single")
    
    mpath = os.path.join(input_dir, f"{tgt}_model.pkl")
    fpath = os.path.join(input_dir, f"{tgt}_feat.pkl")
    model = joblib.load(mpath) if os.path.exists(mpath) else None
    feats = None
    if os.path.exists(fpath):
        try:
            feats = joblib.load(fpath)
        except Exception:
            feats = None
    
    end_timer("model_loading_single")
    return model, feats

def _fallback_feature_order_for_model(model):
    """Generate fallback feature order when metadata is missing."""
    n = getattr(model, "n_features_", None)
    if not isinstance(n, (int, np.integer)) or n < 1:
        return (FEATURE_BASE + ['target_name_encoded'])
    take = max(0, n - 1)
    return (FEATURE_BASE[:take] + ['target_name_encoded'])

def _lazy_load_models():
    """Lazy loading of all models to avoid memory issues during initialization."""
    global MODELS_LOADED, models, MODEL_FEATURES
    if MODELS_LOADED:
        return
    
    start_timer("lazy_load_all_models")
    loaded = 0
    for tgt in TARGETS:
        model, feats = _load_model_and_meta(tgt, MODEL_INPUT_DIR)
        models[tgt] = model
        if (model is not None) and (feats is None):
            feats = _fallback_feature_order_for_model(model)
        MODEL_FEATURES[tgt] = feats if feats is not None else (FEATURE_BASE + ['target_name_encoded'])
        if models[tgt] is not None:
            loaded += 1
    MODELS_LOADED = True
    print(f"[Info] Lazy-loaded models: {loaded} / {len(TARGETS)}")
    end_timer("lazy_load_all_models")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

# Model Training Pipeline
if TRAIN:
    print("\n======== TRAINING MODE ========")
    start_timer("training_preparation")
    
    # Prepare features and targets for training
    X_all_df = create_features(train_df).to_pandas()
    X_all = X_all_df[FEATURE_BASE]
    y_all = train_labels.to_pandas()[TARGETS]
    
    end_timer("training_preparation")
    
    start_timer("training_all_targets")
    
    # Train individual LightGBM model for each target
    for idx, tgt in enumerate(TARGETS):
        start_timer("training_single_target")
        
        print(f"Training {tgt}...")
        y = y_all[tgt]
        mask = ~y.isna()
        X_tr_base = X_all.loc[mask].copy()
        y_tr = y.loc[mask]
        
        # Skip targets with no valid training data
        if len(y_tr) == 0:
            print(f"  Skip {tgt}: no non-null labels.")
            models[tgt] = None
            MODEL_FEATURES[tgt] = FEATURE_BASE + ['target_name_encoded']
            joblib.dump(MODEL_FEATURES[tgt], _meta_path_for(tgt))
            end_timer("training_single_target")
            continue
        
        # Add target-specific encoding and preprocess
        X_tr = X_tr_base.copy()
        X_tr['target_name_encoded'] = idx
        X_tr = preprocess_for_lgbm(X_tr)
        
        feat_order = FEATURE_BASE + ['target_name_encoded']
        X_tr = X_tr.reindex(columns=feat_order, fill_value=0.0)
        
        # Train LightGBM model with optimized hyperparameters
        start_timer("lgbm_fit_single")
        m = LGBMRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1
        )
        m.fit(X_tr, y_tr)
        train_pred = m.predict(X_tr)
        
        train_r2 = r2_score(y_tr, train_pred)
        train_mse = mean_squared_error(y_tr, train_pred)
        
        perf_tracker.record_training_score(tgt, train_r2, 'r2')
        perf_tracker.record_training_score(tgt, train_mse, 'mse')
        perf_tracker.model_sizes[tgt] = perf_tracker.get_model_size(m)
        
        print(f"  {tgt}: R2={train_r2:.4f}, MSE={train_mse:.4f}, Size={perf_tracker.model_sizes[tgt]:.2f}MB")

        end_timer("lgbm_fit_single")
        
        # Store model and feature metadata
        start_timer("model_saving_single")
        models[tgt] = m
        MODEL_FEATURES[tgt] = feat_order
        joblib.dump(m, os.path.join(MODEL_OUTPUT_DIR, f"{tgt}_model.pkl"))
        joblib.dump(feat_order, _meta_path_for(tgt))
        end_timer("model_saving_single")
        
        end_timer("training_single_target")
    
    end_timer("training_all_targets")
    print(f"Saved models & metadata to: {MODEL_OUTPUT_DIR}")
    
    # Plot training histograms
    plot_timing_histogram("training_single_target", "Time per Target Training", bins=30)
    plot_timing_histogram("lgbm_fit_single", "LightGBM Fit Time per Target", bins=30)

In [ ]:
def analyze_model_differences():
    """Compare models across different metrics"""
    print("\n======== MODEL COMPARISON ANALYSIS ========")
    
    # Prepare comparison data
    comparison_data = []
    for tgt in TARGETS[:50]:  # Analyze first 50 targets
        if tgt in perf_tracker.training_scores and models.get(tgt) is not None:
            model = models[tgt]
            comparison_data.append({
                'target': tgt,
                'train_r2': perf_tracker.training_scores[tgt].get('r2', 0),
                'train_mse': perf_tracker.training_scores[tgt].get('mse', 0),
                'cv_sharpe': np.mean(perf_tracker.cv_scores_per_target[tgt]) if tgt in perf_tracker.cv_scores_per_target else 0,
                'model_size_mb': perf_tracker.model_sizes.get(tgt, 0),
                'n_features': model.n_features_ if hasattr(model, 'n_features_') else 0,
                'n_leaves': model.num_leaves
            })
    
    df_comparison = pd.DataFrame(comparison_data)
    
    # Correlation analysis
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: R2 vs CV Sharpe
    axes[0, 0].scatter(df_comparison['train_r2'], df_comparison['cv_sharpe'], alpha=0.6)
    axes[0, 0].set_xlabel('Training R2')
    axes[0, 0].set_ylabel('CV Sharpe')
    axes[0, 0].set_title('Training R2 vs CV Performance')
    
    # Plot 2: Model Size Distribution
    axes[0, 1].hist(df_comparison['model_size_mb'], bins=20, edgecolor='black')
    axes[0, 1].set_xlabel('Model Size (MB)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Model Size Distribution')
    
    # Plot 3: Performance by Target Index
    axes[0, 2].plot(df_comparison['cv_sharpe'].rolling(10).mean(), label='10-target MA')
    axes[0, 2].set_xlabel('Target Index')
    axes[0, 2].set_ylabel('CV Sharpe (MA)')
    axes[0, 2].set_title('Performance Trend Across Targets')
    axes[0, 2].legend()
    
    # Plot 4: Feature Count vs Performance
    axes[1, 0].scatter(df_comparison['n_features'], df_comparison['cv_sharpe'], alpha=0.6)
    axes[1, 0].set_xlabel('Number of Features')
    axes[1, 0].set_ylabel('CV Sharpe')
    axes[1, 0].set_title('Feature Count vs Performance')
    
    # Plot 5: MSE Distribution
    axes[1, 1].hist(np.log1p(df_comparison['train_mse']), bins=20, edgecolor='black')
    axes[1, 1].set_xlabel('Log(MSE + 1)')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Training MSE Distribution (log scale)')
    
    # Plot 6: Top/Bottom Performers
    top_10 = df_comparison.nlargest(10, 'cv_sharpe')
    bottom_10 = df_comparison.nsmallest(10, 'cv_sharpe')
    axes[1, 2].barh(range(10), top_10['cv_sharpe'].values, color='green', alpha=0.7, label='Top 10')
    axes[1, 2].barh(range(10, 20), bottom_10['cv_sharpe'].values, color='red', alpha=0.7, label='Bottom 10')
    axes[1, 2].set_yticks(list(range(10)) + list(range(10, 20)))
    axes[1, 2].set_yticklabels(list(top_10['target'].values) + list(bottom_10['target'].values), fontsize=8)
    axes[1, 2].set_xlabel('CV Sharpe')
    axes[1, 2].set_title('Best vs Worst Performing Targets')
    axes[1, 2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Statistical summary
    print("\nModel Performance Statistics:")
    print(df_comparison[['train_r2', 'train_mse', 'cv_sharpe', 'model_size_mb']].describe())
    
    # Identify problematic models
    print("\nPotentially Overfitting Models (high train R2, low CV):")
    overfit = df_comparison[(df_comparison['train_r2'] > 0.7) & (df_comparison['cv_sharpe'] < 0.1)]
    print(overfit[['target', 'train_r2', 'cv_sharpe']].head(10))
    
    return df_comparison

In [ ]:
if TRAIN:
    model_comparison_df = analyze_model_differences()

In [ ]:
# Feature Importance Analysis
feature_importance_dict = {}
print("\n======== FEATURE IMPORTANCE ANALYSIS ========")

for idx, tgt in enumerate(TARGETS[:5]):  # Analyze first 5 targets as examples
    if models.get(tgt) is not None:
        model = models[tgt]
        feat_order = MODEL_FEATURES[tgt]
        importance = model.feature_importances_
        feature_importance_dict[tgt] = pd.DataFrame({
            'feature': feat_order,
            'importance': importance
        }).sort_values('importance', ascending=False)

# Plot top features across models
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, tgt in enumerate(list(feature_importance_dict.keys())[:3]):
    df = feature_importance_dict[tgt].head(20)
    axes[i].barh(range(len(df)), df['importance'])
    axes[i].set_yticks(range(len(df)))
    axes[i].set_yticklabels(df['feature'], fontsize=8)
    axes[i].set_title(f'Top 20 Features - {tgt}')
    axes[i].set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Aggregate feature importance across all models
all_importance = pd.DataFrame()
for tgt, df in feature_importance_dict.items():
    df['target'] = tgt
    all_importance = pd.concat([all_importance, df])

avg_importance = all_importance.groupby('feature')['importance'].mean().sort_values(ascending=False)
print("\nTop 10 Most Important Features (averaged across targets):")
print(avg_importance.head(10))

In [ ]:
# Cross-Validation with Time Series Split
if DO_CV:
    print(f"\nRunning TimeSeriesSplit CV with {CV_SPLITS} folds...")
    start_timer("cross_validation_total")
    
    SOL_FILL = SOLUTION_NULL_FILLER
    X_base = X_all.copy()
    y_truth = y_all.copy()
    y_score = y_truth.fillna(SOL_FILL)
    
    # Initialize out-of-fold predictions
    oof = pd.DataFrame(index=y_truth.index, columns=TARGETS, dtype=float)
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    fold_scores = []
    
    # Perform time series cross-validation
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_base)):
        start_timer("cv_single_fold")
        
        print(f"  CV Fold {fold+1}/{CV_SPLITS}")
        X_tr_b, X_va_b = X_base.iloc[tr_idx], X_base.iloc[va_idx]
        y_tr_all = y_truth.iloc[tr_idx]
        
        # Train model for each target on this fold
        for idx, tgt in enumerate(TARGETS):
            start_timer("cv_single_target_training")
            
            y_tr_tgt = y_tr_all[tgt]
            mask_tr = ~y_tr_tgt.isna()
            
            if mask_tr.sum() == 0:
                end_timer("cv_single_target_training")
                continue
            
            # Prepare fold-specific training data
            Xtr = X_tr_b.loc[mask_tr].copy()
            ytr = y_tr_tgt.loc[mask_tr]
            Xva = X_va_b.copy()
            Xtr['target_name_encoded'] = idx
            Xva['target_name_encoded'] = idx
            
            Xtr = preprocess_for_lgbm(Xtr).reindex(columns=(FEATURE_BASE + ['target_name_encoded']), fill_value=0.0)
            Xva = preprocess_for_lgbm(Xva).reindex(columns=(FEATURE_BASE + ['target_name_encoded']), fill_value=0.0)
            
            # Train fold-specific model
            m_cv = LGBMRegressor(
                n_estimators=80,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=64,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbose=-1
            )
            m_cv.fit(Xtr, ytr)
            oof.loc[va_idx, tgt] = m_cv.predict(Xva)
            
            end_timer("cv_single_target_training")
        
        # Calculate fold score using competition metric
        fold_score = rankcorr_sharpe(
            preds_df=oof.loc[va_idx, TARGETS],
            truths_df=y_score.loc[va_idx, TARGETS],
            filler=SOL_FILL
        )
        fold_scores.append(fold_score)
        print(f"   -> Fold {fold+1} Sharpe: {fold_score:.4f}")

        end_timer("cv_single_fold")

        target_scores = []
        for tgt in TARGETS:
            if not oof.loc[va_idx, tgt].isna().all():
                tgt_score = rankcorr_sharpe(
                    oof.loc[va_idx, [tgt]], 
                    y_score.loc[va_idx, [tgt]], 
                    SOL_FILL
                )
                target_scores.append(tgt_score)
                perf_tracker.cv_scores_per_target[tgt].append(tgt_score)

    print(f"   -> Fold {fold+1} Target Score Distribution: mean={np.mean(target_scores):.4f}, std={np.std(target_scores):.4f}")
    
    # Calculate overall cross-validation score
    cv_score = rankcorr_sharpe(oof, y_score, SOL_FILL)
    print(f"  OOF RankCorr-Sharpe = {cv_score:.4f}")
    if len(fold_scores) > 0:
        print(f"  Mean(Fold Sharpe)   = {np.mean(fold_scores):.4f}")
    
    end_timer("cross_validation_total")
    
    # Plot CV histograms
    plot_timing_histogram("cv_single_fold", "Time per CV Fold", bins=10)
    plot_timing_histogram("cv_single_target_training", "CV Target Training Time", bins=30)
else:
    print("CV disabled (DO_CV=False).")

In [ ]:
if DO_CV:
    model_comparison_df = analyze_model_differences()

In [ ]:
# Public Leaderboard Simulation
if DO_PUBLIC90:
    print("\nRunning Public-90 style hold-out (last ~90 days)...")
    pub90_score = validate_public_90_style(train_df, train_labels)
    print(f"  Public-90 Sharpe     = {pub90_score:.4f}")
    
    # Plot Public-90 histograms
    plot_timing_histogram("public90_single_target_training", "Public-90 Target Training Time", bins=30)
else:
    print("\n======== INFERENCE MODE ========")
    print(f"Models will be lazy-loaded from: {MODEL_INPUT_DIR}")

In [ ]:
if DO_PUBLIC90 and DO_CV and 'cv_score' in locals():
    print(f"\nScore Comparison:")
    print(f"  CV OOF Score: {cv_score:.4f}")
    print(f"  Public-90 Score: {pub90_score:.4f}")
    print(f"  Difference (Public90 - CV): {pub90_score - cv_score:.4f}")
    if abs(pub90_score - cv_score) > 0.2:
        print("  ⚠️ Large discrepancy between CV and Public-90 scores detected")

In [ ]:
def analyze_prediction_performance():
    """Analyze prediction timing and characteristics"""
    if not perf_tracker.prediction_times:
        return
        
    print("\n======== PREDICTION PERFORMANCE ANALYSIS ========")
    
    pred_times = []
    for tgt, times in perf_tracker.prediction_times.items():
        if times:
            pred_times.append(np.mean(times))
    
    if pred_times:
        print(f"Average prediction time per target: {np.mean(pred_times):.6f}s")
        print(f"Total prediction time for all targets: {np.sum(pred_times):.3f}s")
        print(f"Slowest target predictions: {np.max(pred_times):.6f}s")
        print(f"Fastest target predictions: {np.min(pred_times):.6f}s")

In [ ]:
# Information Coefficient (IC) - add in validation functions
def calculate_ic(preds, truths):
    """Calculate Information Coefficient (Spearman correlation)"""
    from scipy.stats import spearmanr
    mask = ~(np.isnan(preds) | np.isnan(truths))
    if mask.sum() < 2:
        return 0
    return spearmanr(preds[mask], truths[mask])[0]

# Maximum Drawdown for predictions - useful for financial data
def calculate_max_drawdown(returns):
    """Calculate maximum drawdown from returns series"""
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min()

# Calmar Ratio (Sharpe-like but uses max drawdown)
def calculate_calmar_ratio(returns):
    """Calculate Calmar ratio: mean return / max drawdown"""
    mean_return = returns.mean()
    max_dd = abs(calculate_max_drawdown(returns))
    return mean_return / max_dd if max_dd > 0 else 0

In [ ]:
# Leak-Safe Prediction Function for Kaggle Server
def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pd.DataFrame:
    """
    Main prediction function called by Kaggle evaluation server.
    Implements strict leak prevention and shape safety:
    - Uses only intersection-based features
    - Adds target-specific encoding
    - Handles missing models gracefully
    - Ensures exact column order and no NaN values
    """
    start_timer("prediction_total")
    
    _lazy_load_models()
    
    # Apply feature engineering and select base features
    df_feat = create_features(test)
    X_base = df_feat.to_pandas()
    base_selected = X_base.reindex(columns=FEATURE_BASE, fill_value=0.0)
    
    # Initialize prediction array
    n_rows = len(base_selected)
    out = np.zeros((n_rows, len(TARGETS)), dtype=np.float64)
    
    # Generate predictions for each target
    for idx, tgt in enumerate(TARGETS):
        start_timer("prediction_single_target")
        
        model = models.get(tgt)
        if model is None:
            out[:, idx] = 0.0
            end_timer("prediction_single_target")
            continue
        
        # Get feature order used during training
        feat_order = MODEL_FEATURES.get(tgt)
        if feat_order is None:
            feat_order = _fallback_feature_order_for_model(model)
        
        # Prepare features with target encoding
        X_tmp = base_selected.copy()
        X_tmp['target_name_encoded'] = idx
        
        # Preprocess and align with training feature order
        X_tmp = preprocess_for_lgbm(X_tmp)
        X_tmp = X_tmp.reindex(columns=feat_order, fill_value=0.0)
        
        # Generate prediction and handle shape mismatches
        pred = model.predict(X_tmp, validate_features=False)
        pred = np.asarray(pred, dtype=np.float64).reshape(-1)

        perf_tracker.prediction_times[tgt].append(time.time() - start_pred_time)

        if pred.shape[0] != n_rows:
            if pred.shape[0] > n_rows:
                pred = pred[:n_rows]
            else:
                pred = np.pad(pred, (0, n_rows - pred.shape[0]), constant_values=0.0)
        
        out[:, idx] = pred
        end_timer("prediction_single_target")

    
    if pred_times:
        print(f"Average prediction time per target: {np.mean(pred_times):.6f}s")
        print(f"Total prediction time for all targets: {np.sum(pred_times):.3f}s")
        print(f"Slowest target predictions: {np.max(pred_times):.6f}s")
        print(f"Fastest target predictions: {np.min(pred_times):.6f}s")
    
    # Ensure no infinite or NaN values in final predictions
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Create final prediction DataFrame with strict validation
    predictions = pd.DataFrame(out, columns=TARGETS)
    assert predictions.shape[1] == 424, f"Expected 424 columns, got {predictions.shape[1]}"
    assert list(predictions.columns) == TARGETS, "Column order/name mismatch"
    assert np.isfinite(predictions.to_numpy()).all(), "Non-finite values in predictions"
    
    end_timer("prediction_total")
    return predictions

In [ ]:
# Main Execution and Server Launch
print("Pipeline execution complete!")

if IS_KAGGLE:
    # Initialize Kaggle evaluation server
    server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)
    if IS_COMP_RUN:
        server.serve()
    else:
        server.run_local_gateway((DATA_PATH,))
else:
    # Local testing mode
    mock = pl.DataFrame()
    submission = predict(test_df, mock, mock, mock, mock)
    submission = submission[TARGETS]
    submission.to_csv("submission_local.csv", index=False)
    print("Local submission saved → submission_local.csv")

In [ ]:
if not IS_KAGGLE:
    analyze_prediction_performance()

In [ ]:
# Diagnostic Utilities
def validate_public_90_style_leaky(train_pl: pl.DataFrame, labels_pl: pl.DataFrame) -> float:
    """
    Diagnostic validation that trains on ALL available data then evaluates on last 90 days.
    This provides an upper bound estimate and helps understand potential overfitting.
    """
    start_timer("leaky_validation")
    
    X_all_df = create_features(train_pl).to_pandas()
    X_all = X_all_df[FEATURE_BASE].copy()
    y_all = labels_pl.to_pandas()[TARGETS].copy()
    N, H = len(X_all), 90
    
    if N <= H:
        print("[Leaky Public-90] Not enough length; skipped.")
        end_timer("leaky_validation")
        return 0.0
    
    va_idx = np.arange(N - H, N)
    preds_va = pd.DataFrame(index=range(H), columns=TARGETS, dtype=float)
    sol_va = y_all.iloc[va_idx].reset_index(drop=True)
    
    for idx, tgt in enumerate(TARGETS):
        start_timer("leaky_single_target_training")
        
        y = y_all[tgt]
        msk = ~y.isna()
        if msk.sum() == 0:
            preds_va[tgt] = 0.0
            end_timer("leaky_single_target_training")
            continue
        
        # Train on ALL available data (leaky)
        Xtr = X_all.loc[msk].copy(); ytr = y.loc[msk]
        Xva = X_all.iloc[va_idx].copy()
        Xtr["target_name_encoded"] = idx
        Xva["target_name_encoded"] = idx
        
        Xtr = preprocess_for_lgbm(Xtr).reindex(columns=(FEATURE_BASE+["target_name_encoded"]), fill_value=0.0)
        Xva = preprocess_for_lgbm(Xva).reindex(columns=(FEATURE_BASE+["target_name_encoded"]), fill_value=0.0)
        
        m = LGBMRegressor(
            n_estimators=100, learning_rate=0.05,
            max_depth=-1, num_leaves=64, subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbose=-1
        )
        m.fit(Xtr, ytr)
        preds_va[tgt] = m.predict(Xva)
        
        end_timer("leaky_single_target_training")
    
    result = rankcorr_sharpe(preds_va, sol_va, filler=SOLUTION_NULL_FILLER)
    end_timer("leaky_validation")
    return result

# Run diagnostic leaky validation if in training mode
if TRAIN and DO_PUBLIC90 and not IS_COMP_RUN:
    print("\n[Diag] Leaky Public-90 style (train=full → val=last 90)...")
    leaky_score = validate_public_90_style_leaky(train_df, train_labels)
    print(f"  Leaky Public-90 Sharpe = {leaky_score:.4f}  (Reference value: tends to be closer to the public leaderboard score)")

In [ ]:
if 'pub90_score' in locals() and 'cv_score' in locals():
    print(f"\n  Validation Score Summary:")
    print(f"    CV Score: {cv_score:.4f}")
    print(f"    Public-90 Score: {pub90_score:.4f}")  
    print(f"    Leaky Public-90: {leaky_score:.4f}")
    print(f"    Gap (Leaky - Clean): {leaky_score - pub90_score:.4f}")

In [ ]:
# Model Artifact Packaging
start_timer("model_packaging")

# Configuration for model packaging
SRC_DIR = Path("/kaggle/working/model")        # directory containing trained model files
OUT_ZIP = Path("/kaggle/working/model_bundle.zip")
INCLUDE_EXTS = {".pkl"}                        # file extensions to include in archive

# Collect all model files recursively
start_timer("file_collection")
files = [p for p in SRC_DIR.rglob("*") if p.is_file() and p.suffix.lower() in INCLUDE_EXTS]
files.sort()
end_timer("file_collection")

if not files:
    raise SystemExit(f"No files with extensions {INCLUDE_EXTS} found under {SRC_DIR}")
print(f"Found {len(files)} file(s) to zip from: {SRC_DIR}")

# Create compressed archive of all model files
start_timer("zip_creation")
with zipfile.ZipFile(OUT_ZIP, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for i, fp in enumerate(files, 1):
        # Preserve relative paths within archive
        arcname = fp.relative_to(SRC_DIR)
        zf.write(fp, arcname)
        if i % 50 == 0 or i == len(files):
            print(f"  Added {i}/{len(files)} -> {arcname}")
end_timer("zip_creation")
print("\nZip build complete.")

# Verify archive integrity and display summary
start_timer("zip_verification")
with zipfile.ZipFile(OUT_ZIP, mode="r") as zf:
    bad_member = zf.testzip()
    if bad_member is None:
        print("Integrity check: OK (no corrupt members).")
    else:
        print(f"Integrity check: CORRUPT member found -> {bad_member}")
end_timer("zip_verification")

# Display archive statistics
size_mb = OUT_ZIP.stat().st_size / (1024 * 1024)
print(f"Archive path : {OUT_ZIP}")
print(f"Archive size : {size_mb:.2f} MB")
with zipfile.ZipFile(OUT_ZIP, mode="r") as zf:
    names = zf.namelist()
    head = names[:10]
    print("\nPreview (first 10 entries):")
    for n in head:
        print(f"  - {n}")
    print(f"... total entries: {len(names)}")
end_timer("model_packaging")

In [ ]:
# Comprehensive Timing Analysis and Visualization
# Calculate total pipeline time
total_pipeline_time = time.time() - pipeline_start
print(f"\n" + "="*80)
print(f"PIPELINE TIMING SUMMARY")
print(f"="*80)
print(f"Total Pipeline Time: {total_pipeline_time:.2f} seconds ({total_pipeline_time/60:.2f} minutes)")

def plot_all_remaining_histograms():
    """Plot histograms for all timing data that hasn't been plotted yet"""
    
    # Define which histograms were already plotted
    already_plotted = {
        "training_single_target", 
        "lgbm_fit_single", 
        "cv_single_fold", 
        "cv_single_target_training", 
        "public90_single_target_training"
    }
    
    # Get all available timing categories
    all_categories = list(timing_data.keys())
    remaining_categories = [cat for cat in all_categories if cat not in already_plotted]
    
    print(f"\nPlotting histograms for {len(remaining_categories)} remaining timing categories...")
    
    # Sort categories by importance/frequency
    category_priority = {
        "preprocess_for_lgbm_single": 1,
        "create_features_single": 2,
        "rankcorr_sharpe_single": 3,
        "model_saving_single": 4,
        "model_loading_single": 5,
        "prediction_single_target": 6,
        "leaky_single_target_training": 7,
        "data_loading": 8,
        "data_sorting": 9,
        "setup": 10,
        "training_preparation": 11,
        "training_all_targets": 12,
        "cross_validation_total": 13,
        "validate_public_90_style": 14,
        "prediction_total": 15,
        "leaky_validation": 16,
        "lazy_load_all_models": 17,
        "model_packaging": 18,
        "file_collection": 19,
        "zip_creation": 20,
        "zip_verification": 21
    }
    
    remaining_categories.sort(key=lambda x: category_priority.get(x, 999))
    
    for category in remaining_categories:
        if len(timing_data[category]) > 0:
            plot_timing_histogram(category, f"Time Distribution: {category.replace('_', ' ').title()}")

def display_timing_summary():
    """Display a comprehensive summary of all timing data"""
    
    print(f"\n" + "="*80)
    print(f"DETAILED TIMING BREAKDOWN")
    print(f"="*80)
    
    # Sort categories by total time spent
    category_totals = []
    for category, times in timing_data.items():
        if len(times) > 0:
            total_time = np.sum(times)
            mean_time = np.mean(times)
            count = len(times)
            category_totals.append((category, total_time, mean_time, count))
    
    category_totals.sort(key=lambda x: x[1], reverse=True)  # Sort by total time
    
    print(f"{'Category':<35} {'Count':<8} {'Total (s)':<12} {'Mean (s)':<12} {'% of Total':<10}")
    print("-" * 80)
    
    for category, total_time, mean_time, count in category_totals:
        percentage = (total_time / total_pipeline_time) * 100
        print(f"{category:<35} {count:<8} {total_time:<12.3f} {mean_time:<12.6f} {percentage:<10.2f}%")
    
    # Identify bottlenecks
    print(f"\n" + "="*50)
    print(f"PERFORMANCE BOTTLENECKS")
    print(f"="*50)
    
    # Top 5 by total time
    print("\nTop 5 processes by total time:")
    for i, (category, total_time, mean_time, count) in enumerate(category_totals[:5]):
        print(f"  {i+1}. {category}: {total_time:.3f}s ({(total_time/total_pipeline_time)*100:.1f}%)")
    
    # Top 5 by mean time (for repeated operations)
    repeated_ops = [(cat, tot, mean, cnt) for cat, tot, mean, cnt in category_totals if cnt > 1]
    repeated_ops.sort(key=lambda x: x[2], reverse=True)
    
    if repeated_ops:
        print("\nSlowest repeated operations (by mean time):")
        for i, (category, total_time, mean_time, count) in enumerate(repeated_ops[:5]):
            print(f"  {i+1}. {category}: {mean_time:.6f}s avg ({count} calls)")
    
    # Memory and efficiency insights
    print(f"\n" + "="*50)
    print(f"EFFICIENCY INSIGHTS")
    print(f"="*50)
    
    # Calculate preprocessing overhead
    preprocess_times = timing_data.get("preprocess_for_lgbm_single", [])
    if preprocess_times:
        total_preprocess = np.sum(preprocess_times)
        print(f"Data preprocessing overhead: {total_preprocess:.3f}s ({len(preprocess_times)} calls)")
    
    # Calculate model training vs other operations
    training_times = timing_data.get("lgbm_fit_single", [])
    if training_times:
        total_training = np.sum(training_times)
        training_pct = (total_training / total_pipeline_time) * 100
        print(f"Pure model training time: {total_training:.3f}s ({training_pct:.1f}% of total)")
    
    # Calculate I/O overhead
    io_categories = ["data_loading", "model_saving_single", "model_loading_single", "zip_creation"]
    total_io = sum(np.sum(timing_data.get(cat, [])) for cat in io_categories)
    io_pct = (total_io / total_pipeline_time) * 100
    print(f"I/O operations time: {total_io:.3f}s ({io_pct:.1f}% of total)")

def analyze_performance_patterns():
    """Analyze performance patterns and potential optimizations"""
    
    print(f"\n" + "="*50)
    print(f"PERFORMANCE PATTERN ANALYSIS")
    print(f"="*50)
    
    # Analyze target training time variance
    target_training_times = timing_data.get("training_single_target", [])
    if len(target_training_times) > 10:
        variance = np.var(target_training_times)
        std_dev = np.std(target_training_times)
        mean_time = np.mean(target_training_times)
        cv = std_dev / mean_time  # Coefficient of variation
        
        print(f"\nTarget Training Time Analysis:")
        print(f"  Coefficient of Variation: {cv:.3f}")
        if cv > 0.5:
            print(f"  ⚠️  High variance detected - some targets much slower than others")
        elif cv < 0.2:
            print(f"  ✅ Low variance - consistent training times across targets")
        else:
            print(f"  ℹ️  Moderate variance in training times")
    
    # Analyze preprocessing efficiency
    preprocess_times = timing_data.get("preprocess_for_lgbm_single", [])
    if len(preprocess_times) > 10:
        preprocess_mean = np.mean(preprocess_times)
        training_mean = np.mean(timing_data.get("lgbm_fit_single", [0]))
        
        if training_mean > 0:
            preprocess_ratio = preprocess_mean / training_mean
            print(f"\nPreprocessing Efficiency:")
            print(f"  Preprocessing vs Training ratio: {preprocess_ratio:.3f}")
            if preprocess_ratio > 0.3:
                print(f"  ⚠️  Preprocessing overhead is high relative to training")
            else:
                print(f"  ✅ Preprocessing overhead is reasonable")
    
    # Memory usage patterns (estimated)
    model_save_times = timing_data.get("model_saving_single", [])
    if len(model_save_times) > 10:
        save_variance = np.var(model_save_times)
        if save_variance > np.mean(model_save_times) ** 2:
            print(f"\nModel Saving Analysis:")
            print(f"  ⚠️  High variance in model save times - possible memory pressure")

# Display comprehensive timing analysis
display_timing_summary()
analyze_performance_patterns()

# Plot all remaining histograms
plot_all_remaining_histograms()

In [ ]:
# Final summary plot: Overview of main pipeline stages
def plot_pipeline_overview():
    """Create an overview plot of main pipeline stages"""
    
    # Define main pipeline stages and their components
    pipeline_stages = {
        "Setup & Loading": ["setup", "data_loading", "data_sorting", "feature_definition"],
        "Training": ["training_preparation", "training_all_targets"],
        "Cross Validation": ["cross_validation_total"],
        "Public-90 Validation": ["validate_public_90_style"],
        "Model Packaging": ["model_packaging", "file_collection", "zip_creation", "zip_verification"]
    }
    
    stage_times = []
    stage_labels = []
    
    for stage_name, components in pipeline_stages.items():
        total_stage_time = 0
        for component in components:
            if component in timing_data and len(timing_data[component]) > 0:
                total_stage_time += np.sum(timing_data[component])
        
        if total_stage_time > 0:
            stage_times.append(total_stage_time)
            stage_labels.append(stage_name)
    
    if stage_times:
        plt.figure(figsize=(12, 8))
        
        # Create pie chart
        plt.subplot(2, 1, 1)
        colors = plt.cm.Set3(np.linspace(0, 1, len(stage_times)))
        wedges, texts, autotexts = plt.pie(stage_times, labels=stage_labels, autopct='%1.1f%%', 
                                      colors=colors, startangle=90)
        plt.title('Pipeline Time Distribution by Stage')
        
        # Create bar chart
        plt.subplot(2, 1, 2)
        bars = plt.bar(range(len(stage_times)), stage_times, color=colors)
        plt.xlabel('Pipeline Stage')
        plt.ylabel('Time (seconds)')
        plt.title('Pipeline Stage Times')
        plt.xticks(range(len(stage_labels)), stage_labels, rotation=45, ha='right')
        
        # Add value labels on bars
        for bar, time_val in zip(bars, stage_times):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stage_times)*0.01,
                    f'{time_val:.1f}s', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nPipeline Stage Summary:")
        for stage, time_val in zip(stage_labels, stage_times):
            percentage = (time_val / total_pipeline_time) * 100
            print(f"  {stage}: {time_val:.2f}s ({percentage:.1f}%)")

plot_pipeline_overview()
print(f"\n" + "="*80)
print(f"TIMING ANALYSIS COMPLETE")
print(f"="*80)
print(f"Total categories tracked: {len(timing_data)}")
print(f"Total measurements taken: {sum(len(times) for times in timing_data.values())}")
print(f"Pipeline completed in: {total_pipeline_time:.2f} seconds")
print(f"="*80)

In [ ]:
def create_performance_dashboard():
    """Create a comprehensive performance dashboard"""
    print("\n" + "="*80)
    print("COMPREHENSIVE MODEL PERFORMANCE DASHBOARD")
    print("="*80)
    
    # Summary statistics
    if perf_tracker.training_scores:
        r2_scores = [v.get('r2', 0) for v in perf_tracker.training_scores.values()]
        print(f"\nTraining R2: mean={np.mean(r2_scores):.4f}, std={np.std(r2_scores):.4f}")
    
    if perf_tracker.cv_scores_per_target:
        all_cv_scores = []
        for scores in perf_tracker.cv_scores_per_target.values():
            all_cv_scores.extend(scores)
        print(f"CV Sharpe: mean={np.mean(all_cv_scores):.4f}, std={np.std(all_cv_scores):.4f}")
    
    # Model efficiency
    if perf_tracker.model_sizes:
        total_size = sum(perf_tracker.model_sizes.values())
        print(f"\nTotal model size: {total_size:.2f} MB")
        print(f"Average model size: {total_size/len(perf_tracker.model_sizes):.2f} MB")
    
    # Timing efficiency
    print(f"\nPipeline efficiency: {len(TARGETS) / (total_pipeline_time/60):.2f} models/minute")

create_performance_dashboard()